# Extracción de Características Temporales (Feature Engineering) para sEMG

Este cuadernillo se encarga de realizar la **extracción manual de características (Feature Engineering)** a partir de las ventanas temporales de señal sEMG monocanal de tu prototipo Myotensor.

Extraeremos **5 características fundamentales en el dominio del tiempo (Time Domain Features)** recomendadas en la literatura científica para clasificar gestos utilizando algoritmos tradicionales como **Support Vector Machine (SVM)** y **Random Forest (RF)**:

1. **MAV** (Mean Absolute Value)
2. **RMS** (Root Mean Square)
3. **WL** (Waveform Length)
4. **ZC** (Zero Crossings)
5. **SSC** (Slope Sign Changes)

## Carga de Datos

In [6]:
import numpy as np
import pandas as pd
import os
import joblib

# Función puramente robusta para buscar y cargar el archivo .env
def load_env_variables():
    import os
    from pathlib import Path
    
    # 1. Buscar .env subiendo niveles desde el CWD actual
    try:
        start_dir = Path(os.getcwd())
    except:
        start_dir = Path(".")
        
    env_path = None
    for path in [start_dir] + list(start_dir.parents):
        temp_path = path / ".env"
        if temp_path.exists():
            env_path = temp_path
            break
            
    if env_path is None:
        raise FileNotFoundError("⚠️ No se pudo encontrar el archivo .env en la raíz del proyecto.")
        
    # 2. Leer e inyectar variables en os.environ
    with open(env_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                continue
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip()
            
    print(f"✅ Archivo .env cargado con éxito desde: {env_path}")

load_env_variables()

✅ Archivo .env cargado con éxito desde: /home/cbe/Proyectos/MyoTensor_Tesis/.env


## Definición de Funciones (Time Domain Features)

In [7]:
# 1. Definición de Fórmulas Matemáticas y Funciones Vectorizadas en NumPy

def mean_absolute_value(x):
    """
    Mean Absolute Value (MAV):
    Representa la energía promedio de la contracción muscular en la ventana.
    Fórmula: MAV = (1/W) * sum(|x_i|)
    """
    return np.mean(np.abs(x))

def root_mean_square(x):
    """
    Root Mean Square (RMS):
    Mide la potencia media de la señal sEMG, que se correlaciona directamente con la fuerza muscular.
    Fórmula: RMS = sqrt( (1/W) * sum(x_i^2) )
    """
    return np.sqrt(np.mean(np.square(x)))

def waveform_length(x):
    """
    Waveform Length (WL):
    Mide la complejidad de la señal sEMG. Es la suma acumulada de las diferencias absolutas entre muestras consecutivas.
    Fórmula: WL = sum(|x_{i+1} - x_i|)
    """
    return np.sum(np.abs(np.diff(x)))

def zero_crossings(x, threshold=0.005):
    """
    Zero Crossings (ZC):
    Mide la frecuencia de oscilación de la señal estimando cuántas veces cruza el cero.
    Se aplica un umbral (threshold) para ignorar cruces pequeños causados únicamente por ruido de fondo.
    """
    x = np.asarray(x).flatten()
    # Detectar cambios de signo entre muestras consecutivas
    sign_changes = (x[:-1] * x[1:]) < 0
    # Filtrar por umbral de ruido
    above_threshold = np.abs(x[:-1] - x[1:]) > threshold
    return np.sum(sign_changes & above_threshold)

def slope_sign_changes(x, threshold=0.005):
    """
    Slope Sign Changes (SSC):
    Mide los cambios de dirección de la pendiente de la señal.
    Sirve como otro indicador indirecto de la frecuencia dominante.
    Se aplica un umbral en las pendientes consecutivas para evitar falsos positivos por ruido.
    """
    x = np.asarray(x).flatten()
    d1 = x[1:-1] - x[:-2]
    d2 = x[2:] - x[1:-1]
    # Cambios de signo de la pendiente
    slope_changes = (d1 * d2) < 0
    # Filtrar pendientes menores al umbral de ruido
    above_threshold = (np.abs(d1) > threshold) & (np.abs(d2) > threshold)
    return np.sum(slope_changes & above_threshold)

# 2. Carga de los Tensores de Señal y Reconstrucción del Dataset Original
base_path = os.environ["PROCESSED_TENSOR_PROTO"]
print("Cargando tensores procesados de Myotensor Proto...")
X_train = np.load(f'{base_path}/X_train.npy')
X_test  = np.load(f'{base_path}/X_test.npy')
y_train_cat = np.load(f'{base_path}/y_train.npy')
y_test_cat  = np.load(f'{base_path}/y_test.npy')

# Deshacer el escalado a nivel muestra para obtener la señal sEMG real sin escalar
scaler_raw = joblib.load(f'{base_path}/std_scaler.bin')
W = X_train.shape[1]

X_train_raw = scaler_raw.inverse_transform(X_train.reshape(-1, 1)).reshape(X_train.shape[0], W)
X_test_raw  = scaler_raw.inverse_transform(X_test.reshape(-1, 1)).reshape(X_test.shape[0], W)

# Concatenar para formar el dataset completo a transformar
X_data = np.concatenate([X_train_raw, X_test_raw], axis=0)
y_data = np.argmax(np.concatenate([y_train_cat, y_test_cat], axis=0), axis=1)

print(f"✅ X_data reconstruido exitosamente: {X_data.shape} ventanas.")
print(f"✅ y_data reconstruido: {y_data.shape} con clases {np.unique(y_data)}")

Cargando tensores procesados de Myotensor Proto...
✅ X_data reconstruido exitosamente: (2048, 300) ventanas.
✅ y_data reconstruido: (2048,) con clases [0 1 2 3]


## Extracción de Características (Transformación del Tensor)

In [8]:
print("⚡ Iniciando extracción manual de características...")
num_ventanas = X_data.shape[0]
X_features = np.zeros((num_ventanas, 5))

# Umbral de ruido adaptado para las señales monocanal sEMG del Myotensor
noise_threshold = 0.005

for i in range(num_ventanas):
    window_signal = X_data[i]
    X_features[i, 0] = mean_absolute_value(window_signal)
    X_features[i, 1] = root_mean_square(window_signal)
    X_features[i, 2] = waveform_length(window_signal)
    X_features[i, 3] = zero_crossings(window_signal, threshold=noise_threshold)
    X_features[i, 4] = slope_sign_changes(window_signal, threshold=noise_threshold)

print(f"¡Extracción finalizada con éxito!")
print(f"Dimensión final del tensor de características X_features: {X_features.shape}")

# Visualizar las primeras filas en un DataFrame de Pandas para verificación estética y numérica
feature_names = ['MAV', 'RMS', 'WL', 'ZC', 'SSC']
df_preview = pd.DataFrame(X_features, columns=feature_names)
df_preview['Clase'] = y_data
print("\n--- Vista Previa de Características Extraídas ---")
print(df_preview.head())

⚡ Iniciando extracción manual de características...
¡Extracción finalizada con éxito!
Dimensión final del tensor de características X_features: (2048, 5)

--- Vista Previa de Características Extraídas ---
        MAV       RMS         WL    ZC   SSC  Clase
0  0.365016  0.480817  68.632995  61.0  82.0      2
1  0.147541  0.177705  32.858916  71.0  82.0      0
2  0.232626  0.291053  48.228853  64.0  86.0      2
3  0.176710  0.220289  35.601717  59.0  80.0      0
4  0.350455  0.460486  61.858815  54.0  86.0      3


## División de Datos y Estandarización (Vital para SVM)

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Dividiendo el conjunto de características en Train (80%) y Test (20%)...")

# Split estratificado por clases (stratify=y_data) para mantener el balance de gestos
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_features,
    y_data,
    test_size=0.2,
    random_state=42,
    stratify=y_data
)

print(f"   * X_train_f: {X_train_f.shape} | y_train_f: {y_train_f.shape}")
print(f"   * X_test_f:  {X_test_f.shape}  | y_test_f:  {y_test_f.shape}")

print("\n⚡ Ajustando StandardScaler de Scikit-Learn...")
scaler_features = StandardScaler()

# Ajustar (fit) únicamente en entrenamiento para evitar Data Leakage
scaler_features.fit(X_train_f)

# Transformar conjuntos de entrenamiento y prueba
X_train_scaled = scaler_features.transform(X_train_f)
X_test_scaled  = scaler_features.transform(X_test_f)

print(f"✅ Conjunto de Entrenamiento Escalado: {X_train_scaled.shape}")
print(f"✅ Conjunto de Prueba Escalado:       {X_test_scaled.shape}")

Dividiendo el conjunto de características en Train (80%) y Test (20%)...
   * X_train_f: (1638, 5) | y_train_f: (1638,)
   * X_test_f:  (410, 5)  | y_test_f:  (410,)

⚡ Ajustando StandardScaler de Scikit-Learn...
✅ Conjunto de Entrenamiento Escalado: (1638, 5)
✅ Conjunto de Prueba Escalado:       (410, 5)


## Guardado de los Vectores Finales

In [10]:
# Se crea el subdirectorio 'vector_classic' en datasets desde la ruta absoluta .env
output_dir = os.environ["PROCESSED_VECTOR_CLASSIC_PROTO"]
os.makedirs(output_dir, exist_ok=True)

print(f"💾 Guardando matrices resultantes en: {output_dir} ...")
np.save(os.path.join(output_dir, 'X_train_scaled.npy'), X_train_scaled)
np.save(os.path.join(output_dir, 'X_test_scaled.npy'), X_test_scaled)
np.save(os.path.join(output_dir, 'y_train.npy'), y_train_f)
np.save(os.path.join(output_dir, 'y_test.npy'), y_test_f)

# Guardar también el escalador de características
scaler_path = os.path.join(output_dir, 'features_scaler.bin')
joblib.dump(scaler_features, scaler_path)

print("\n🎉 ¡Todo listo y guardado con éxito! Matrices listas para entrenar SVM y Random Forest.")

💾 Guardando matrices resultantes en: /home/cbe/Proyectos/MyoTensor_Tesis/intelligence/datasets/processed/myotensor_proto/vector_classic ...

🎉 ¡Todo listo y guardado con éxito! Matrices listas para entrenar SVM y Random Forest.
